# Chapter 3 — Build It Yourself: The N-gram MLP

The hands-on heart of [Chapter 3](../README.md). You'll build a neural language model with
**embeddings** and a **context** of 3 characters, train it on tensors, and watch it beat
the bigram. **✍️ Your turn** cells (hint + "Stuck? Show the answer"), **📖 Study & run**
cells for the mechanical parts, **▶️ Check your work** cells throughout. Top to bottom. 🚀

## Step 1 — Setup, data, vocabulary  ▶️ Run this

In [ ]:
from pathlib import Path
import random
import torch
import torch.nn.functional as F

DATA = next(p for p in [Path("../data/names.txt"), Path("data/names.txt"),
                        Path("chapters/03-ngram-mlp/data/names.txt")] if p.exists())
words = [w.strip() for w in DATA.read_text().splitlines() if w.strip()]
chars = sorted(set("".join(words)))
stoi = {c: i + 1 for i, c in enumerate(chars)}; stoi["."] = 0
itos = {i: c for c, i in stoi.items()}
vocab_size = len(stoi)
block_size = 3
print(f"{len(words)} names | vocab {vocab_size} | block_size {block_size}")

## Step 2 — Build the dataset with context  ✍️ Your turn

We slide a window of `block_size` characters over each name (padded with `.` = 0) and
record *(context → next char)*. Fill in the line that **slides the window**: drop the
oldest id and append the new one, `ix`.

<details><summary>Stuck? Show the answer</summary>

```python
context = context[1:] + [ix]
```
</details>

In [ ]:
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            # ✍️ slide the window forward: drop the oldest id, append the new one (ix)
            context = context
    return torch.tensor(X), torch.tensor(Y)

random.seed(42); random.shuffle(words)
n1, n2 = int(0.8 * len(words)), int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
print("Xtr", tuple(Xtr.shape), "| Ytr", tuple(Ytr.shape))

In [ ]:
# ▶️ Check your work
try:
    assert Xtr.shape[1] == 3, f"each context should be 3 ids, got shape {tuple(Xtr.shape)}"
    assert Xtr.shape[0] == Ytr.shape[0], "X and Y must have the same number of rows"
    assert int(Xtr.sum()) > 0, "every context is all-zero — did the window actually slide?"
    _Xa, _ = build_dataset(['a'])     # 'a' gives contexts [0,0,0] then [0,0,stoi['a']]
    assert _Xa[1].tolist() == [0, 0, stoi['a']], "the window slid the wrong way — the new id goes on the RIGHT (context[1:] + [ix])"
    print(f"✅ Dataset built — {Xtr.shape[0]} training examples, each a context of 3 ids.")
except (NameError, AssertionError) as e:
    print("❌", e)

## Step 3 — The parameters  📖 Study & run

The embedding table `C` plus a two-layer MLP. The output weights start tiny (`* 0.01`) so
the first predictions are near-uniform and the loss begins around `log(27) ≈ 3.3` instead
of exploding — initialization is its own topic (Chapter 7).

In [ ]:
g = torch.Generator().manual_seed(2147483647)
n_embd, n_hidden = 10, 200
C  = torch.randn((vocab_size, n_embd), generator=g)                 # embedding table (27, 10)
W1 = torch.randn((block_size * n_embd, n_hidden), generator=g) * 0.2 # hidden layer (30, 200)
b1 = torch.randn(n_hidden, generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01         # output layer (200, 27)
b2 = torch.zeros(vocab_size)
params = [C, W1, b1, W2, b2]
for p in params:
    p.requires_grad = True
print(sum(p.nelement() for p in params), "parameters")

## Step 4 — Look up the embeddings  ✍️ Your turn

**Fancy indexing**, built up: `C[5]` returns one embedding `(10,)`; `C[[5, 7]]` returns two
stacked `(2, 10)`; and `C[Xb]` with `Xb` of shape `(32, 3)` returns `(32, 3, 10)` — *32
pages, each a 3 × 10 grid* (for every example, its 3 characters, each now a 10-vector).

**Your task:** index `C` with `Xb`, then **flatten** each example's `3 × 10` block into one
row of `30` numbers (`.view`, with `-1` to infer the size).

<details><summary>Stuck? Show the answer</summary>

```python
emb = C[Xb]                          # (32, 3, 10)
x = emb.view(emb.shape[0], -1)       # (32, 30)
```
</details>

In [ ]:
ix = torch.randint(0, Xtr.shape[0], (32,), generator=g)   # a minibatch of 32 examples
Xb, Yb = Xtr[ix], Ytr[ix]

# ✍️ look up embeddings for the batch, then flatten each example to one row
emb = Xb        # ✍️ replace: index the embedding table C with the batch Xb
x = emb         # ✍️ replace: flatten each example into one row (use .view; -1 infers the size)
print("emb", tuple(emb.shape), "-> x", tuple(x.shape))

In [ ]:
# ▶️ Check your work
try:
    assert tuple(emb.shape) == (32, block_size, n_embd), f"emb should be (32, 3, 10), got {tuple(emb.shape)} — index C with Xb"
    assert tuple(x.shape) == (32, block_size * n_embd), f"x should be (32, 30) after flattening, got {tuple(x.shape)}"
    print(f"✅ Looked up and flattened — each of the 32 examples is now {x.shape[1]} numbers.")
except (NameError, AssertionError) as e:
    print("❌", e)

## Step 5 — The forward pass and the loss  ✍️ Your turn

Push `x` through the hidden layer (`tanh`), then the output layer to get 27 **logits**, then
score them with **`cross_entropy`** (softmax + negative-log-likelihood, in one call).

<details><summary>Stuck? Show the answer</summary>

```python
h = torch.tanh(x @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Yb)
```
</details>

In [ ]:
h = x          # ✍️ hidden layer: x times W1, plus b1, squashed by tanh
logits = h     # ✍️ output layer: h times W2, plus b2  → the 27 logits
loss = None    # ✍️ cross-entropy of the logits against the targets Yb (one F.cross_entropy call)
print("logits shape:", tuple(logits.shape), "| loss:",
      round(loss.item(), 4) if loss is not None else "fill me in")

In [ ]:
# ▶️ Check your work
try:
    assert tuple(logits.shape) == (32, vocab_size), f"logits should be (32, 27), got {tuple(logits.shape)}"
    assert loss is not None and 2.5 < loss.item() < 4.0, f"untrained loss should be near log(27)=3.3, got {loss}"
    print(f"✅ Forward pass works — 27 logits per example, loss {loss.item():.3f} (≈ log(27), as expected before training).")
except (NameError, AttributeError, AssertionError) as e:
    print("❌", e)

## Step 6 — Train it  📖 Study & run

The forward → backward → update loop, on tensors, with minibatches. ~20k steps, a few
seconds. Watch the minibatch loss wobble downward (the wobble is just minibatch noise), and
read the honest **dev** loss at the end. (We run 20k steps here for speed; the script
`code/mlp.py` runs 30k and reaches ~2.15.)

In [ ]:
def forward(X):
    emb = C[X]
    h = torch.tanh(emb.view(emb.shape[0], -1) @ W1 + b1)
    return h @ W2 + b2

for step_i in range(20000):
    ix = torch.randint(0, Xtr.shape[0], (32,), generator=g)
    loss = F.cross_entropy(forward(Xtr[ix]), Ytr[ix])
    for p in params:
        p.grad = None
    loss.backward()
    lr = 0.1 if step_i < 12000 else 0.01
    for p in params:
        p.data += -lr * p.grad
    if step_i % 4000 == 0:
        print(f"step {step_i:5d} | minibatch loss {loss.item():.4f}")

@torch.no_grad()
def split_loss(X, Y):
    return F.cross_entropy(forward(X), Y).item()

print(f"\ntrain {split_loss(Xtr, Ytr):.4f} | dev {split_loss(Xdev, Ydev):.4f}  (bigram was 2.45)")

In [ ]:
# ▶️ Check your work
try:
    dev = split_loss(Xdev, Ydev)
    assert dev < 2.4, f"dev loss should beat the bigram's 2.45, got {dev:.4f}"
    print(f"✅ Trained! dev loss {dev:.4f} — well below the bigram's 2.45, and it generalizes.")
except (NameError, AssertionError) as e:
    print("❌", e)

## Step 7 — Generate names  ✍️ Your turn

The autoregressive loop, now through the MLP: get the next-char probabilities, **sample**
one, and **slide** the context window forward. (The model expects a *batch*, so we wrap the
single context as `torch.tensor([context])` — one row — and `dim=1` softmaxes across the 27
columns.)

<details><summary>Stuck? Show the answer</summary>

```python
ix = torch.multinomial(probs, num_samples=1, generator=gen).item()
context = context[1:] + [ix]
```
</details>

In [ ]:
@torch.no_grad()
def sample(gen):
    out, context = [], [0] * block_size
    while True:
        probs = F.softmax(forward(torch.tensor([context])), dim=1)
        # ✍️ sample the next id from probs, then slide the context window
        ix = 0                     # ✍️ sample the next id from probs (a torch.multinomial call, then .item())
        context = context          # ✍️ slide the window: drop the oldest id, append the new ix
        if ix == 0:
            break
        out.append(itos[ix])
    return "".join(out)

gen = torch.Generator().manual_seed(2147483647 + 10)
for _ in range(12):
    print(" ", sample(gen))

In [ ]:
# ▶️ Check your work
try:
    gen2 = torch.Generator().manual_seed(7)
    names = [sample(gen2) for _ in range(20)]
    assert sum(len(n) > 0 for n in names) >= 15, "most names are empty — did you sample (multinomial) AND slide the context?"
    assert all(all(c in itos.values() for c in n) for n in names), "names should be made of valid characters"
    sample_str = ", ".join(n for n in names if n)[:60]
    print(f"✅ Your MLP generates names! e.g. {sample_str}…")
except (NameError, AssertionError) as e:
    print("❌", e)

## 🎓 You built a neural language model.

Embeddings, a context of 3, an MLP, minibatch training — dev loss ~2.15, and names that
actually look like names. This token → embedding → layers → logits → softmax shape is the
skeleton of every LLM.

**Next:** the [exercises](../exercises/) (tune it, *plot the embeddings*, try GELU), the
[mini-project](../project/) (*Name Forge MK II*), then
[Chapter 4 — Attention](../../04-attention/). 👋